# 04 — Hybrid Search

Build and evaluate image, semantic, and hybrid search using FAISS, BM25, and precomputed embeddings.

## 1 · Install packages

In [1]:
%pip -q install "transformers>=4.44,<5" "faiss-cpu>=1.8" "rank-bm25>=0.2" pandas pyarrow pillow requests tqdm

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.30.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.0 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [6]:
import faiss
import transformers
import huggingface_hub
from rank_bm25 import BM25Okapi

print("FAISS:", faiss.__version__)
print("Transformers:", transformers.__version__)
print("HuggingFace Hub:", huggingface_hub.__version__)
print("All imports successful!")

FAISS: 1.15.1
Transformers: 4.57.6
HuggingFace Hub: 0.36.2
All imports successful!


## 2 · Imports and Drive

In [2]:
import re, json, pickle, time, hashlib
from pathlib import Path
from io import BytesIO
import numpy as np
import pandas as pd
import faiss
from rank_bm25 import BM25Okapi
import torch
import torch.nn.functional as F
from PIL import Image, ImageOps, UnidentifiedImageError
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer, AutoModel
from google.colab import drive
import requests

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/ElectroLens")
OUT = ROOT / "search_assets"
CLIP_ID = "openai/clip-vit-base-patch32"
E5_ID = "intfloat/multilingual-e5-small"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


Mounted at /content/drive
Device: cuda


## 3 · Load and validate saved embeddings

In [3]:
products = pd.read_parquet(OUT / "products_search.parquet").reset_index(drop=True)
manifest = json.loads((OUT / "manifest.json").read_text())
asins = products["parent_asin"].astype(str).to_numpy()
actual_hash = hashlib.sha256("\n".join(asins).encode()).hexdigest()[:16]
if actual_hash != manifest["catalog_hash"] or len(products) != manifest["n_products"]:
    raise ValueError("Catalog does not match embedding manifest")
if manifest["clip_model"] != CLIP_ID or manifest["e5_model"] != E5_ID:
    raise ValueError("Model IDs do not match saved embeddings")
e5_vectors = np.load(OUT / "e5_embeddings.npy", mmap_mode="r")
image_vectors = np.load(OUT / "clip_image_embeddings.npy", mmap_mode="r")
image_valid = np.load(OUT / "clip_image_valid.npy")
if e5_vectors.shape != (len(products), 384) or image_vectors.shape != (len(products), 512) or image_valid.shape != (len(products),):
    raise ValueError("Embedding array dimensions do not match catalog")
print("Products:", len(products), "| valid images:", int(image_valid.sum()))


Products: 13328 | valid images: 13328


## 4 · Reconstruct product text for BM25

In [4]:
def clean_field(x, limit=350):
    if x is None or isinstance(x, (list, dict)): return ""
    s = re.sub(r"\s+", " ", str(x)).strip()
    return "" if s.lower() in {"nan", "none", "null"} else s[:limit]

def make_product_text(row):
    fields = [clean_field(row.get("title"), 180)]
    for name in ("store", "product_group", "description"):
        if name in products.columns:
            value = clean_field(row.get(name), 250)
            if value: fields.append(value)
    return " | ".join(fields)

product_texts = products.apply(make_product_text, axis=1).tolist()
print(product_texts[0])

TOZO T10 Bluetooth 5.3 Wireless Earbuds with Wireless Charging Case IPX8 Waterproof Stereo Headphones in Ear Built in Mic Headset Premium Sound with Deep Bass for Sport White (2022 | TOZO | headphones


## 5 · Build and save FAISS indexes

In [7]:
CLIP_DIM = image_vectors.shape[1]
E5_DIM = e5_vectors.shape[1]

image_rows = np.flatnonzero(image_valid).astype("int64")
if len(image_rows) == 0:
    raise RuntimeError("No image embeddings created. Inspect image URLs/network.")

image_index = faiss.IndexFlatIP(CLIP_DIM)
image_index.add(np.ascontiguousarray(image_vectors[image_rows]))

e5_index = faiss.IndexFlatIP(E5_DIM)
e5_index.add(np.ascontiguousarray(e5_vectors))

faiss.write_index(image_index, str(OUT / "clip_image.faiss"))
faiss.write_index(e5_index, str(OUT / "e5_text.faiss"))
np.save(OUT / "clip_image_row_ids.npy", image_rows)

print("CLIP image index:", image_index.ntotal, "| E5 index:", e5_index.ntotal)

CLIP image index: 13328 | E5 index: 13328


## 6 · Build and save BM25

In [8]:
def tokenize(s):
    return re.findall(r"\w+", str(s).lower(), flags=re.UNICODE)

tokenized_products = [tokenize(t) for t in product_texts]
bm25 = BM25Okapi(tokenized_products)
with open(OUT / "bm25.pkl", "wb") as f:
    pickle.dump(bm25, f, protocol=pickle.HIGHEST_PROTOCOL)
print("BM25 ready:", len(tokenized_products), "documents")

BM25 ready: 13328 documents


## 7 · Load query encoders (once per app process)

This notebook loads CLIP and E5 only to encode **new queries**. In production, initialize models and indexes at FastAPI startup. First request can be slower due to warm-up.

In [9]:
clip = CLIPModel.from_pretrained(CLIP_ID).to(DEVICE).eval()
clip_processor = CLIPProcessor.from_pretrained(CLIP_ID)
e5_tokenizer = AutoTokenizer.from_pretrained(E5_ID)
e5 = AutoModel.from_pretrained(E5_ID).to(DEVICE).eval()
CLIP_DIM = int(clip.config.projection_dim)
E5_DIM = int(e5.config.hidden_size)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

## 8 · Query embedding functions

In [10]:
@torch.inference_mode()
def encode_clip_images(images):
    x = clip_processor(images=images, return_tensors="pt").to(DEVICE)
    z = clip.get_image_features(**x)
    return F.normalize(z.float(), p=2, dim=1).cpu().numpy().astype("float32")

@torch.inference_mode()
def encode_clip_text(texts):
    x = clip_processor(text=texts, padding=True, truncation=True, max_length=77,
                       return_tensors="pt").to(DEVICE)
    z = clip.get_text_features(**x)
    return F.normalize(z.float(), p=2, dim=1).cpu().numpy().astype("float32")

@torch.inference_mode()
def encode_e5(texts, prefix="passage: "):
    batch = e5_tokenizer([prefix + t for t in texts], padding=True, truncation=True,
                         max_length=256, return_tensors="pt").to(DEVICE)
    out = e5(**batch).last_hidden_state
    mask = batch["attention_mask"].unsqueeze(-1)
    z = (out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    return F.normalize(z.float(), p=2, dim=1).cpu().numpy().astype("float32")

## 9 · Text search: BM25 + E5 + RRF

In [11]:
def e5_search(query, k=100):
    q = encode_e5([query], prefix="query: ")
    scores, idx = e5_index.search(q, min(k, e5_index.ntotal))
    return [int(i) for i in idx[0] if i >= 0]

def bm25_search(query, k=100):
    scores = bm25.get_scores(tokenize(query))
    # Stable ranking by score, then index.
    order = np.argsort(-scores, kind="stable")[:min(k, len(scores))]
    return [int(i) for i in order]

def rrf_fuse(rankings, top_k=10, rrf_k=60):
    fused = {}
    for ranked in rankings:
        for rank, row in enumerate(ranked, 1):
            fused[row] = fused.get(row, 0.0) + 1.0 / (rrf_k + rank)
    return sorted(fused, key=lambda row: (-fused[row], row))[:top_k]

def hybrid_search(query, top_k=10, candidate_k=100):
    if not query.strip(): return products.iloc[[]].copy()
    rows = rrf_fuse([e5_search(query, candidate_k),
                     bm25_search(query, candidate_k)], top_k=top_k)
    result = products.iloc[rows].copy()
    result.insert(0, "catalog_row", rows)
    return result.reset_index(drop=True)

display(hybrid_search("wireless gaming headphones", top_k=5)[["parent_asin", "title"]])

,parent_asin,title
0,B092H1NQ19,occiam Bluetooth Headphones 4-Mics Call Waterp...
1,B0BSLYLP42,Wireless Earbuds Bluetooth Headphones 60H Play...
2,B09ZTGG152,Hikapa X9 Wireless Earbuds Bluetooth 5.3 Headp...
3,B0BBQ6F2TK,Midola Gaming Bluetooth 5.0 Wireless Cat Ear H...
4,B0C6SW75CX,Caymuller Wireless Earbuds Bluetooth Headphone...


## 10 · Image search and CLIP text→image

In [12]:
def fetch_image(url):
    if not isinstance(url, str) or not url.startswith(("https://", "http://")):
        return None
    try:
        r = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        if len(r.content) > 12_000_000: return None
        with Image.open(BytesIO(r.content)) as im:
            img = ImageOps.exif_transpose(im).convert("RGB")
            img.thumbnail((1024, 1024))
            return img.copy()
    except (requests.RequestException, OSError, UnidentifiedImageError, ValueError):
        return None

def fetch_batch(urls):
    with ThreadPoolExecutor(max_workers=IMAGE_WORKERS) as pool:
        return list(pool.map(fetch_image, urls))

In [15]:
from PIL import Image, ImageOps, UnidentifiedImageError

def clip_image_search(image, top_k=10):
    q = encode_clip_images([image])
    scores, ids = image_index.search(q, min(top_k, image_index.ntotal))
    rows = image_rows[ids[0]]
    result = products.iloc[rows].copy()
    result.insert(0, "similarity", scores[0])
    return result.reset_index(drop=True)

def clip_text_to_image_search(query, top_k=10):
    q = encode_clip_text([query])
    scores, ids = image_index.search(q, min(top_k, image_index.ntotal))
    rows = image_rows[ids[0]]
    result = products.iloc[rows].copy()
    result.insert(0, "similarity", scores[0])
    return result.reset_index(drop=True)

display(clip_text_to_image_search("black over ear headphones", 5)[["title", "similarity"]])

sample_row = int(image_rows[0])
sample_img = fetch_image(products.iloc[sample_row].image_url)
if sample_img is not None:
    display(clip_image_search(sample_img, 5)[["title", "similarity"]])

,title,similarity
0,"Sony MDRRF985RK Wireless RF Headphone, Black",0.332659
1,Razer Adaro Stereos - Analog Headphones,0.328541
2,Mackie MC-250 MC Series Professional Studio He...,0.326699
3,"Waterproof Over-Ear Headphones, V5.0 HD Stereo...",0.326607
4,Audio-Technica ATH-AVC200 SonicPro Over-Ear Cl...,0.325804


,title,similarity
0,TOZO T10 Bluetooth 5.3 Wireless Earbuds with W...,1.000000
1,TOZO T6 True Wireless Earbuds Bluetooth 5.3 He...,0.929951
2,TOZO NC9 2022 Version Hybrid Active Noise Canc...,0.910150
3,5.2 Bluetooth Headphones for iPhone 14 Plus 13...,0.895194
4,SoundPEATS Air3 Wireless Earbuds Mini Bluetoot...,0.886715


## 11 · Measure warm query latency

In [16]:
def time_call(fn, *args, repeats=5):
    fn(*args)
    times = []
    for _ in range(repeats):
        t0 = time.perf_counter()
        fn(*args)
        if DEVICE == "cuda": torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return {"median_ms": round(float(np.median(times)), 1),
            "p95_ms": round(float(np.percentile(times, 95)), 1)}

query = "noise cancelling bluetooth earbuds"
latencies = {
    "BM25 only": time_call(bm25_search, query),
    "E5 encode + FAISS": time_call(e5_search, query),
    "Hybrid BM25 + E5 + RRF": time_call(hybrid_search, query),
    "CLIP text + image FAISS": time_call(clip_text_to_image_search, query),
}
if sample_img is not None:
    latencies["CLIP image + image FAISS"] = time_call(clip_image_search, sample_img)
display(pd.DataFrame(latencies).T)

,median_ms,p95_ms
BM25 only,11.6,13.4
E5 encode + FAISS,12.1,12.2
Hybrid BM25 + E5 + RRF,29.1,32.3
CLIP text + image FAISS,13.0,20.0
CLIP image + image FAISS,22.1,22.7


## 12 · Verify saved search artifacts

In [17]:
required_files = [
    "products_search.parquet", "clip_image.faiss", "clip_image_row_ids.npy",
    "e5_text.faiss", "bm25.pkl", "manifest.json",
]
for name in required_files:
    path = OUT / name
    print(f"{name:28s} {'OK' if path.exists() else 'MISSING':8s} {path.stat().st_size / (1024**2):7.2f} MB" if path.exists() else name + " MISSING")
print("\nOutput folder:", OUT)

products_search.parquet      OK         12.55 MB
clip_image.faiss             OK         26.03 MB
clip_image_row_ids.npy       OK          0.10 MB
e5_text.faiss                OK         19.52 MB
bm25.pkl                     OK          5.04 MB
manifest.json                OK          0.00 MB

Output folder: /content/drive/MyDrive/ElectroLens/search_assets


In [18]:
queries = [
    "wireless noise cancelling headphones",
    "Sony Bluetooth headphones",
    "Logitech wireless gaming mouse",
    "portable power bank for charging phone",
    "camera for recording YouTube videos",
    "laptop stand for working at desk",
    "USB-C fast charging cable",
    "cheap wireless earbuds",
    "telefon üçün simsiz qulaqlıq",
    "headphones for listening to music while running"
]

for query in queries:
    print(f"\nQUERY: {query}")
    display(hybrid_search(query, top_k=5)[["title", "price"]])


QUERY: wireless noise cancelling headphones


,title,price
0,Sony WH-H910N h.ear on 3 Wireless Noise-Cancel...,129.00
1,JBL Tour ONE Wireless Noise Cancelling Bluetoo...,174.99
2,Bose Noise Cancelling Wireless Bluetooth Headp...,349.95
3,Bose QuietComfort 45 Bluetooth Wireless Noise ...,329.00
4,Srhythm NC25 Wireless Headphones Bluetooth 5.3...,59.99



QUERY: Sony Bluetooth headphones


,title,price
0,Sony WF-XB700 Truly Wireless Bluetooth Headpho...,139.99
1,Sony WI-C100 Wireless in-Ear Bluetooth Headpho...,20.36
2,Sony WI-SP600N Black Premium Waterproof Blueto...,119.95
3,Sony WH-910N Wireless Bluetooth Headphones Noi...,149.00
4,"Sony MDR-1000X Noise Cancelling, Bluetooth Hea...",434.01



QUERY: Logitech wireless gaming mouse


...

,title,price
0,"EarFun Wireless Earbuds, [2020 CES Award] Free...",39.99
1,SONTINH CoolBuds2 True Wireless Bluetooth Earb...,39.99
2,Tempo 30 Wireless Earbuds for Small Ears Women...,49.99
3,"EarFun Air True Wireless Earbuds, Bluetooth Ea...",42.39
4,Bluetooth Earbuds 5.0 Wireless Earbuds CVC8.0 ...,32.99



QUERY: telefon üçün simsiz qulaqlıq


,title,price
0,"kurdene Bluetooth Wireless Earbuds, S8 Deep Ba...",20.99
1,Sony WI-XB400 Wireless In-Ear Extra Bass Heads...,49.99
2,Samsung Eo-Eg920Bw White Headset/Handsfree/Hea...,8.98
3,Sony WF-1000XM3 Industry Leading Noise Canceli...,149.00
4,TOZO T10 Bluetooth 5.3 Wireless Earbuds with W...,21.99



QUERY: headphones for listening to music while running


,title,price
0,"BESIGN SH03 Sports Bluetooth 4.1 Headphones, W...",27.99
1,"WANFEI Bone Conduction Headphones, Running Blu...",49.99
2,"Rovking Sport Headphones Wired Sweatproof, Ove...",11.99
3,"ZZN Bluetooth Running Headphones, Wireless Wat...",9.99
4,"GGMM Bluetooth Headphones, In Ear Wireless Ear...",12.99


In [19]:
checks = [
    "power bank",
    "gaming mouse",
    "laptop stand",
    "usb-c"
]

for term in checks:
    matches = products[
        products["title"].str.contains(
            term, case=False, na=False, regex=False
        )
    ]

    print(f"\n{term}: {len(matches)} products")
    display(matches[["title", "price"]].head(5))


power bank: 15 products


,title,price
11096,"W-KING Bluetooth Speaker, 50W Powerful Bluetoo...",79.99
11119,W-KING Portable Loud Bluetooth Speakers with S...,71.99
11121,W-KING Portable Loud Bluetooth Speakers with S...,89.99
11180,W-KING Portable Loud Bluetooth Speakers with S...,115.99
11235,Soundcore Anker Motion Boom Plus IP67 Outdoor ...,179.99



gaming mouse: 19 products


,title,price
10039,Rii RM200 2.4G Wireless Mouse with USB Nano Re...,12.99
10054,Logitech G502 Proteus Spectrum RGB Tunable Gam...,74.00
10149,"Logitech G403 Hero 25K Gaming Mouse, Lightsync...",37.99
10273,Corsair Glaive PRO - RGB Gaming Mouse - Comfor...,30.00
10314,2.4GHz 3D Car Shape Wireless Optical Mouse USB...,9.99



laptop stand: 0 products


,title,price



usb-c: 210 products


,title,price
37,Soundcore by Anker Life A1 True Wireless Earbu...,49.99
58,"Motast Wireless Earbuds, Bluetooth 5.1 Earbuds...",35.99
87,"EarFun Wireless Earbuds, [2020 CES Award] Free...",39.99
108,"EarFun Air True Wireless Earbuds, Bluetooth Ea...",42.39
168,Soundcore by Anker Life P2 Mini True Wireless ...,34.99


In [20]:
query = "Logitech wireless gaming mouse"

for name, search_fn in [
    ("BM25", bm25_search),
    ("E5", e5_search)
]:
    rows = search_fn(query, k=10)

    print(f"\n{name} results:")
    display(products.iloc[rows][["title", "price"]])


BM25 results:


,title,price
10676,Logitech G900 Chaos Spectrum Professional Grad...,70.77
10078,Logitech Wireless Mouse M185 (Swift Grey),11.99
10457,Logitech M187 Wireless Mini Mouse,9.99
10001,Logitech 910-002235 Wireless Mouse M185,13.99
10368,Logitech M310 red Full Size Wireless Mouse,14.95
10572,Logitech® Wireless Mouse M325 (Nature Jewelry),36.99
10352,Logitech G Optical Gaming Mouse G400 with High...,89.99
10061,Logitech MX Master 3 Advanced Wireless Mouse,99.95
10623,Logitech 910-004267 m317 Wireless Mouse Limite...,99.99
10868,Logitech Precision Pro Wireless Mouse,36.99



E5 results:


,title,price
10961,"Logitech Wireless Mouse, Silver",24.99
10301,Logitech M305 Wireless Mouse (Silver),99.99
10794,Logitech Wireless Mouse M305 (Pink Balance) - ...,39.99
10802,Logitech Wireless Mouse M325 - Optical Ambidex...,22.95
10658,Logitech Wireless Mouse M305 (Victorian Wallpa...,99.95
10799,Logitech Wireless Mouse M305 (Vivid Violet),129.99
10319,Logitech Wireless Mouse M185 - Silver,15.19
10037,Logitech M510 Wireless Computer Mouse – Comfor...,29.99
10587,Logitech Wireless Mouse M560 - Light Silver,89.99
11013,Logitech Wireless Mouse M310 - Black Topography,49.99


In [21]:
def hybrid_search_v2(query, top_k=10, candidate_k=100):
    if not query.strip():
        return products.iloc[[]].copy()

    bm25_rows = bm25_search(query, k=candidate_k)
    e5_rows = e5_search(query, k=candidate_k)

    # RRF candidate ranking
    rows = rrf_fuse(
        [bm25_rows, e5_rows],
        top_k=candidate_k * 2
    )

    query_tokens = set(tokenize(query))

    scores = []

    for row in rows:
        title_tokens = set(tokenize(products.iloc[row]["title"]))

        # How many query words appear in the product title?
        coverage = len(query_tokens & title_tokens) / max(
            len(query_tokens), 1
        )

        scores.append((row, coverage))

    # Prioritize title matching, preserve RRF order for ties
    ranked_rows = [
        row for row, score in sorted(
            scores,
            key=lambda x: -x[1]
        )
    ][:top_k]

    result = products.iloc[ranked_rows].copy()
    result.insert(0, "catalog_row", ranked_rows)

    return result.reset_index(drop=True)

In [22]:
queries = [
    "Logitech wireless gaming mouse",
    "portable power bank for charging phone",
    "USB-C fast charging cable",
    "wireless noise cancelling headphones"
]

for query in queries:
    print(f"\nQUERY: {query}")

    print("OLD:")
    display(hybrid_search(query, top_k=5)[["title", "price"]])

    print("NEW:")
    display(hybrid_search_v2(query, top_k=5)[["title", "price"]])


QUERY: Logitech wireless gaming mouse
OLD:


,title,price
0,Logitech M310 red Full Size Wireless Mouse,14.95
1,"Logitech M325 Mouse, Wireless Cocktail, (Cockt...",19.99
2,Logitech M305 Wireless Mouse (Silver),99.99
3,Logitech 910-002235 Wireless Mouse M185,13.99
4,Logitech Precision Pro Wireless Mouse,36.99


NEW:


,title,price
0,Logitech G900 Chaos Spectrum Professional Grad...,70.77
1,Logitech M310 red Full Size Wireless Mouse,14.95
2,"Logitech M325 Mouse, Wireless Cocktail, (Cockt...",19.99
3,Logitech M305 Wireless Mouse (Silver),99.99
4,Logitech 910-002235 Wireless Mouse M185,13.99



QUERY: portable power bank for charging phone
OLD:


...

,title,price
0,USB Power Charger Cable Compatible with Texas ...,7.95
1,UGREEN Electronic Organizer Travel Cable Organ...,24.99
2,Belkin Thunderbolt 3 Dock Pro w/ Thunderbolt 3...,205.78
3,Belkin Thunderbolt 3 Dock Plus Thunderbolt 3 C...,145.00
4,Tribit Active Noise Cancelling Earbuds with CV...,39.99



QUERY: wireless noise cancelling headphones
OLD:


,title,price
0,Sony WH-H910N h.ear on 3 Wireless Noise-Cancel...,129.00
1,JBL Tour ONE Wireless Noise Cancelling Bluetoo...,174.99
2,Bose Noise Cancelling Wireless Bluetooth Headp...,349.95
3,Bose QuietComfort 45 Bluetooth Wireless Noise ...,329.00
4,Srhythm NC25 Wireless Headphones Bluetooth 5.3...,59.99


NEW:


,title,price
0,JBL Tour ONE Wireless Noise Cancelling Bluetoo...,174.99
1,Bose Noise Cancelling Wireless Bluetooth Headp...,349.95
2,Bose QuietComfort 45 Bluetooth Wireless Noise ...,329.00
3,Srhythm NC25 Wireless Headphones Bluetooth 5.3...,59.99
4,"Bose Noise Cancelling Headphones 700,Bluetooth...",379.00


In [23]:
import re

PRODUCT_TYPES = {
    "power bank": ["power bank", "portable charger", "battery pack"],
    "gaming mouse": ["gaming mouse", "gaming mice", "g502", "g403", "g900"],
    "charging cable": [
        "charging cable", "charger cable",
        "usb c cable", "type c cable", "usb-c cable"
    ],
    "headphones": ["headphones", "headphone", "headset"]
}

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"usb[\s-]?c", "usb c", text)
    text = re.sub(r"type[\s-]?c", "usb c", text)
    return text

def detect_product_type(query):
    q = normalize_text(query)

    if "power bank" in q:
        return "power bank"
    if "gaming" in q and "mouse" in q:
        return "gaming mouse"
    if "cable" in q:
        return "charging cable"
    if "headphone" in q:
        return "headphones"

    return None

def hybrid_search_v3(query, top_k=10, candidate_k=200):
    if not query.strip():
        return products.iloc[[]].copy()

    rows = rrf_fuse(
        [
            bm25_search(query, k=candidate_k),
            e5_search(query, k=candidate_k)
        ],
        top_k=candidate_k * 2
    )

    product_type = detect_product_type(query)
    query_tokens = set(tokenize(normalize_text(query))) - {
        "for", "with", "the", "a", "an", "to"
    }

    ranked = []

    for rank, row in enumerate(rows):
        title = normalize_text(products.iloc[row]["title"])
        title_tokens = set(tokenize(title))

        coverage = len(query_tokens & title_tokens) / max(len(query_tokens), 1)
        rrf_score = 1 / (rank + 1)

        type_match = False
        if product_type:
            type_match = any(
                normalize_text(term) in title
                for term in PRODUCT_TYPES[product_type]
            )

        # Prioritize matching product types
        score = coverage + 0.2 * rrf_score + 2.0 * int(type_match)

        ranked.append((row, score, type_match))

    if product_type:
        ranked = [item for item in ranked if item[2]]

    ranked.sort(key=lambda item: -item[1])
    selected_rows = [row for row, _, _ in ranked[:top_k]]

    result = products.iloc[selected_rows].copy()
    result.insert(0, "catalog_row", selected_rows)

    return result.reset_index(drop=True)

In [24]:
queries = [
    "Logitech wireless gaming mouse",
    "portable power bank for charging phone",
    "USB-C fast charging cable",
    "wireless noise cancelling headphones"
]

for query in queries:
    print(f"\nQUERY: {query}")
    display(hybrid_search_v3(query, top_k=5)[["title", "price"]])


QUERY: Logitech wireless gaming mouse


,title,price
0,Logitech G900 Chaos Spectrum Professional Grad...,70.77
1,Logitech G Optical Gaming Mouse G400 with High...,89.99
2,"Logitech G403 Hero 25K Gaming Mouse, Lightsync...",37.99
3,Superglide - Fastest and Smoothest Mouse Feet/...,24.95
4,Logitech G502 Proteus Spectrum RGB Tunable Gam...,74.00



QUERY: portable power bank for charging phone


,title,price
0,UGREEN Electronic Organizer Travel Cable Organ...,24.99
1,JBL Pulse 4 - Waterproof Portable Bluetooth Sp...,249.90
2,ABFOCE Solar Bluetooth Speaker Portable Outdoo...,59.99
3,"ABFOCE Solar Bluetooth Speaker,IPX7 Waterproof...",69.99
4,Garmin Venu 2 Plus (Slate/Black) GPS Fitness S...,454.99



QUERY: USB-C fast charging cable


,title,price
0,USB Power Charger Cable Compatible with Texas ...,7.95
1,Suptig Charging Cable Type-C Charging Cable 2 ...,9.99
2,TELESIN Triple Charger Battery Charging Box wi...,44.99
3,Suptig Charging Cable Mini USB Charging Cable ...,7.99
4,"SOULWIT Cable Holder Clips, 3-Pack Cable Manag...",6.35



QUERY: wireless noise cancelling headphones


,title,price
0,JBL Tour ONE Wireless Noise Cancelling Bluetoo...,174.99
1,Bose Noise Cancelling Wireless Bluetooth Headp...,349.95
2,Bose QuietComfort 45 Bluetooth Wireless Noise ...,329.00
3,Srhythm NC25 Wireless Headphones Bluetooth 5.3...,59.99
4,"Bose Noise Cancelling Headphones 700,Bluetooth...",379.00


In [25]:
%pip -q install sentence-transformers

In [26]:
from sentence_transformers import CrossEncoder

RERANKER_ID = "cross-encoder/ms-marco-MiniLM-L-6-v2"

reranker = CrossEncoder(RERANKER_ID, device=DEVICE)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [27]:
def hybrid_search_v4(query, top_k=10, candidate_k=100):
    if not query.strip():
        return products.iloc[[]].copy()

    # Retrieve candidates using BM25 + E5
    rows = rrf_fuse(
        [
            bm25_search(query, k=candidate_k),
            e5_search(query, k=candidate_k)
        ],
        top_k=candidate_k * 2
    )

    if not rows:
        return products.iloc[[]].copy()

    # Cross-Encoder evaluates query-product relevance
    pairs = [
        (query, str(products.iloc[row]["title"]))
        for row in rows
    ]

    scores = reranker.predict(
        pairs,
        batch_size=32,
        show_progress_bar=False
    )

    ranked = sorted(
        zip(rows, scores),
        key=lambda x: -float(x[1])
    )[:top_k]

    selected_rows = [int(row) for row, _ in ranked]

    result = products.iloc[selected_rows].copy()
    result.insert(
        0, "relevance_score",
        [float(score) for _, score in ranked]
    )

    return result.reset_index(drop=True)

In [28]:
queries = [
    "Logitech wireless gaming mouse",
    "portable power bank for charging phone",
    "USB-C fast charging cable",
    "wireless noise cancelling headphones"
]

for query in queries:
    print(f"\nQUERY: {query}")

    display(
        hybrid_search_v4(query, top_k=5)[
            ["title", "price", "relevance_score"]
        ]
    )


QUERY: Logitech wireless gaming mouse


,title,price,relevance_score
0,Logitech G900 Chaos Spectrum Professional Grad...,70.77,8.394022
1,Logitech Wireless Mouse M325 with Designed-for...,14.99,7.517304
2,Logitech Wireless Mouse M325 - Optical Ambidex...,22.95,7.489178
3,"Logitech G403 Hero 25K Gaming Mouse, Lightsync...",37.99,7.439591
4,Logitech Wireless Mouse M325 with Designed-For...,14.99,7.427316



QUERY: portable power bank for charging phone


,title,price,relevance_score
0,JBL Charge 5 Waterproof Portable Speaker with ...,149.95,1.211385
1,JBL Pulse 4 - Waterproof Portable Bluetooth Sp...,249.90,0.998218
2,"Alpatronix Bluetooth Speaker 60W (80W Max), IP...",79.95,0.774316
3,"BUGANI Bluetooth Speaker, M83 Waterproof Porta...",69.99,-0.266095
4,UGREEN Electronic Organizer Travel Cable Organ...,24.99,-0.307126



QUERY: USB-C fast charging cable


,title,price,relevance_score
0,"Monster Wireless Earbuds,Super Fast Charge,Blu...",89.99,4.001698
1,"TRANYA X5 Wireless Earbuds, 32H Playtime with ...",39.99,3.957963
2,Belkin Thunderbolt 3 Dock Pro w/ Thunderbolt 3...,205.78,3.502106
3,USB Power Charger Cable Compatible with Texas ...,7.95,3.193636
4,Belkin Thunderbolt 3 Dock Plus Thunderbolt 3 C...,145.00,3.129870



QUERY: wireless noise cancelling headphones


,title,price,relevance_score
0,Bose QuietComfort 45 Bluetooth Wireless Noise ...,329.00,9.175320
1,Sony Noise Cancelling Headphones WHCH710N: Wir...,71.25,9.150332
2,Srhythm NC35 Noise Cancelling Headphones Wirel...,69.99,9.141066
3,Sony WF-1000XM3 Truly Wireless Noise Cancellin...,99.00,9.078148
4,Sony Noise Cancelling Headphones WHCH700N: Wir...,198.00,8.981153


In [29]:
tests = {
    "Logitech wireless gaming mouse": "gaming mouse",
    "portable power bank for charging phone": "power bank",
    "USB-C fast charging cable": "charging cable"
}

for query, term in tests.items():
    bm_rows = bm25_search(query, k=100)
    e5_rows = e5_search(query, k=100)

    rows = list(dict.fromkeys(bm_rows + e5_rows))
    candidates = products.iloc[rows]

    matches = candidates[
        candidates["title"].str.contains(
            term, case=False, na=False, regex=False
        )
    ]

    print(f"\nQUERY: {query}")
    print("Candidates:", len(candidates))
    print("Matching titles:", len(matches))

    display(matches[["title", "price"]].head(10))


QUERY: Logitech wireless gaming mouse
Candidates: 137
Matching titles: 3


,title,price
10352,Logitech G Optical Gaming Mouse G400 with High...,89.99
10149,"Logitech G403 Hero 25K Gaming Mouse, Lightsync...",37.99
10054,Logitech G502 Proteus Spectrum RGB Tunable Gam...,74.00



QUERY: portable power bank for charging phone
Candidates: 179
Matching titles: 13


,title,price
12226,UGREEN Electronic Organizer Travel Cable Organ...,24.99
12160,"ABFOCE Solar Bluetooth Speaker,IPX7 Waterproof...",69.99
11096,"W-KING Bluetooth Speaker, 50W Powerful Bluetoo...",79.99
11328,JBL Pulse 4 - Waterproof Portable Bluetooth Sp...,249.90
11281,"Alpatronix Bluetooth Speaker 60W (80W Max), IP...",79.95
12228,BUBM 3pcs Multi-Functional (S/M/L in One) Elec...,24.99
12214,Sound Caddy Golf Club Bluetooth Speaker and Po...,59.00
11367,"Bluetooth Speakers, W-KING 60W Loud Portable W...",88.99
11738,ABFOCE Solar Bluetooth Speaker Portable Outdoo...,59.99
11235,Soundcore Anker Motion Boom Plus IP67 Outdoor ...,179.99



QUERY: USB-C fast charging cable
Candidates: 159
Matching titles: 8


,title,price
12467,Suptig Charging Cable Type-C Charging Cable 2 ...,9.99
12240,"SOULWIT Cable Holder Clips, 3-Pack Cable Manag...",6.35
6382,PTHTECHUS Smartwatch Magnet Suction Interface ...,9.99
9356,MoKo Type Cover Fit Microsoft Surface Go 3 (20...,40.99
12469,Suptig Charging Cable Mini USB Charging Cable ...,7.99
968,LZYDD Magnetic Charging Cable for AfterShokz A...,12.99
11004,HUYUN New Micro USB Wire Data line/Charging Ca...,13.99
13193,"Auto Vehicle Direct Wire Charger, 12V 24V to 5...",15.69


In [30]:
print(products.columns.tolist())

['parent_asin', 'title', 'main_group', 'subcategory', 'cat_path', 'store', 'price', 'average_rating', 'rating_number', 'features', 'description', 'image_url', 'product_group', 'kw_ok']


In [31]:
cols = [
    "title",
    "subcategory",
    "cat_path",
    "features",
    "description"
]

for col in cols:
    print(f"\n{col}")
    print("Type:", type(products[col].iloc[0]))
    print("Example:", str(products[col].iloc[0])[:300])

print("\nMissing values:")
print(products[cols].isna().sum())


title
Type: <class 'str'>
Example: TOZO T10 Bluetooth 5.3 Wireless Earbuds with Wireless Charging Case IPX8 Waterproof Stereo Headphones in Ear Built in Mic Headset Premium Sound with Deep Bass for Sport White (2022 Upgraded)

subcategory
Type: <class 'str'>
Example: Earbud Headphones

cat_path
Type: <class 'str'>
Example: Electronics > Headphones, Earbuds & Accessories > Headphones & Earbuds > Earbud Headphones

features
Type: <class 'str'>
Example: [ORIGX Acoustic Technology] The professional ORIGX 2.0 technology further optimises the performance of vocals and midrange, delivering more distinctive immersive and realistic sound through clear layers and extremely rich vocal music details. [Bluetooth 5.3] Features with the advanced Bluetooth 5.3 

description
Type: <class 'str'>
Example: 

Missing values:
title          0
subcategory    0
cat_path       0
features       0
description    0
dtype: int64


In [32]:
def build_rerank_text(row):
    title = str(row["title"])
    category = str(row["subcategory"])
    path = str(row["cat_path"])
    features = str(row["features"])[:400]
    description = str(row["description"])[:200]

    return (
        f"Product: {title}. "
        f"Category: {category}. "
        f"Category path: {path}. "
        f"Features: {features}. "
        f"Description: {description}"
    )


rerank_texts = [
    build_rerank_text(row)
    for _, row in products.iterrows()
]


def hybrid_search_v5(query, top_k=10, candidate_k=100):
    if not query.strip():
        return products.iloc[[]].copy()

    rows = rrf_fuse(
        [
            bm25_search(query, k=candidate_k),
            e5_search(query, k=candidate_k)
        ],
        top_k=candidate_k * 2
    )

    if not rows:
        return products.iloc[[]].copy()

    pairs = [
        (query, rerank_texts[row])
        for row in rows
    ]

    scores = reranker.predict(
        pairs,
        batch_size=32,
        show_progress_bar=False
    )

    ranked = sorted(
        zip(rows, scores),
        key=lambda x: -float(x[1])
    )[:top_k]

    selected_rows = [int(row) for row, _ in ranked]

    result = products.iloc[selected_rows].copy()
    result.insert(
        0,
        "relevance_score",
        [float(score) for _, score in ranked]
    )

    return result.reset_index(drop=True)

In [33]:
queries = [
    "Logitech wireless gaming mouse",
    "portable power bank for charging phone",
    "USB-C fast charging cable",
    "wireless noise cancelling headphones"
]

for query in queries:
    print(f"\nQUERY: {query}")

    print("V4:")
    display(
        hybrid_search_v4(query, top_k=5)[
            ["title", "subcategory", "relevance_score"]
        ]
    )

    print("V5:")
    display(
        hybrid_search_v5(query, top_k=5)[
            ["title", "subcategory", "relevance_score"]
        ]
    )


QUERY: Logitech wireless gaming mouse
V4:


,title,subcategory,relevance_score
0,Logitech G900 Chaos Spectrum Professional Grad...,Mice,8.394022
1,Logitech Wireless Mouse M325 with Designed-for...,Mice,7.517304
2,Logitech Wireless Mouse M325 - Optical Ambidex...,Mice,7.489178
3,"Logitech G403 Hero 25K Gaming Mouse, Lightsync...",Mice,7.439591
4,Logitech Wireless Mouse M325 with Designed-For...,Mice,7.427316


V5:


,title,subcategory,relevance_score
0,Logitech G900 Chaos Spectrum Professional Grad...,Mice,8.837153
1,Logitech Wireless Mouse M305 (Pink Balance) - ...,Mice,6.765971
2,Logitech Wireless Mouse M325 with Designed-for...,Mice,6.668023
3,Logitech M317c (Collection) Wireless Optical M...,Mice,6.649322
4,Logitech Wireless Mouse M305 (Dark Fleur),Mice,6.622442



QUERY: portable power bank for charging phone
V4:


...

,title,subcategory,relevance_score
0,USB Power Charger Cable Compatible with Texas ...,Calculator Accessories,4.498078
1,"TRANYA X5 Wireless Earbuds, 32H Playtime with ...",Earbud Headphones,4.320247
2,Belkin Thunderbolt 3 Dock Pro w/ Thunderbolt 3...,Monitors,4.125835
3,Belkin Thunderbolt 3 Dock Plus Thunderbolt 3 C...,Monitors,3.071725
4,Suptig Charging Cable Type-C Charging Cable 2 ...,Cables & Cords,3.060943



QUERY: wireless noise cancelling headphones
V4:


,title,subcategory,relevance_score
0,Bose QuietComfort 45 Bluetooth Wireless Noise ...,Over-Ear Headphones,9.175320
1,Sony Noise Cancelling Headphones WHCH710N: Wir...,Over-Ear Headphones,9.150332
2,Srhythm NC35 Noise Cancelling Headphones Wirel...,Over-Ear Headphones,9.141066
3,Sony WF-1000XM3 Truly Wireless Noise Cancellin...,Earbud Headphones,9.078148
4,Sony Noise Cancelling Headphones WHCH700N: Wir...,Over-Ear Headphones,8.981153


V5:


,title,subcategory,relevance_score
0,Bose QuietComfort 45 Bluetooth Wireless Noise ...,Over-Ear Headphones,9.447828
1,"Qisebin E7 Active Noise Cancelling Headphones,...",Over-Ear Headphones,9.309890
2,"Bose Noise Cancelling Headphones 700,Bluetooth...",Over-Ear Headphones,9.215843
3,Sony WF-1000XM3 Truly Wireless Noise Cancellin...,Earbud Headphones,9.179806
4,Bose Noise Cancelling Wireless Bluetooth Headp...,Over-Ear Headphones,9.124063


## Final Search Evaluation

Evaluate the selected search pipeline using representative product queries.

In [34]:
test_queries = [
    "wireless noise cancelling headphones",
    "Sony Bluetooth headphones",
    "camera for recording YouTube videos",
    "headphones for running",
    "Logitech wireless gaming mouse"
]

for query in test_queries:
    print(f"\nQuery: {query}")
    display(
        hybrid_search(query, top_k=5)[
            ["title", "subcategory", "price"]
        ]
    )


Query: wireless noise cancelling headphones


,title,subcategory,price
0,Sony WH-H910N h.ear on 3 Wireless Noise-Cancel...,Over-Ear Headphones,129.00
1,JBL Tour ONE Wireless Noise Cancelling Bluetoo...,Over-Ear Headphones,174.99
2,Bose Noise Cancelling Wireless Bluetooth Headp...,Over-Ear Headphones,349.95
3,Bose QuietComfort 45 Bluetooth Wireless Noise ...,Over-Ear Headphones,329.00
4,Srhythm NC25 Wireless Headphones Bluetooth 5.3...,Over-Ear Headphones,59.99



Query: Sony Bluetooth headphones


,title,subcategory,price
0,Sony WF-XB700 Truly Wireless Bluetooth Headpho...,Earbud Headphones,139.99
1,Sony WI-C100 Wireless in-Ear Bluetooth Headpho...,Earbud Headphones,20.36
2,Sony WI-SP600N Black Premium Waterproof Blueto...,Earbud Headphones,119.95
3,Sony WH-910N Wireless Bluetooth Headphones Noi...,Over-Ear Headphones,149.00
4,"Sony MDR-1000X Noise Cancelling, Bluetooth Hea...",Earbud Headphones,434.01



Query: camera for recording YouTube videos


,title,subcategory,price
0,OIEXI 4K Camera for YouTube Videos HD Camcorde...,Camcorders,189.95
1,4K Video Camera Camcorder 48MP Ultra HD Video ...,Camcorders,149.99
2,4K Video Camera Camcorder UHD 48MP WiFi IR Nig...,Camcorders,149.99
3,"Video Camera, 1080P FHD Camcorder 30FPS 24MP V...",Camcorders,89.99
4,"papalook Webcam 1080P Full HD PC Skype Camera,...",Webcams,21.99



Query: headphones for running


,title,subcategory,price
0,"WANFEI Bone Conduction Headphones, Running Blu...",Earbud Headphones,49.99
1,"ZZN Bluetooth Running Headphones, Wireless Wat...",Earbud Headphones,9.99
2,"JVC HAECX20B Sports Clip Inner Ear Headphones,...",Earbud Headphones,19.99
3,SHOKZ (AfterShokz OpenMove - Open-Ear Bluetoot...,Earbud Headphones,79.95
4,"GGMM Bluetooth Headphones, In Ear Wireless Ear...",Earbud Headphones,12.99



Query: Logitech wireless gaming mouse


,title,subcategory,price
0,Logitech M310 red Full Size Wireless Mouse,Mice,14.95
1,"Logitech M325 Mouse, Wireless Cocktail, (Cockt...",Mice,19.99
2,Logitech M305 Wireless Mouse (Silver),Mice,99.99
3,Logitech 910-002235 Wireless Mouse M185,Mice,13.99
4,Logitech Precision Pro Wireless Mouse,Mice,36.99


In [35]:
required_files = [
    "e5_text.faiss",
    "clip_image.faiss",
    "clip_image_row_ids.npy",
    "bm25.pkl"
]

for filename in required_files:
    path = OUT / filename
    print(f"{filename}: {'OK' if path.exists() else 'MISSING'}")

e5_text.faiss: OK
clip_image.faiss: OK
clip_image_row_ids.npy: OK
bm25.pkl: OK


## Conclusion

Built a multimodal product search system using CLIP, Multilingual E5-small, FAISS, BM25, and Reciprocal Rank Fusion (RRF).

### Results
- Hybrid text search: 29.1 ms median latency.
- CLIP text-to-image search: 13.0 ms.
- CLIP image-to-image search: 22.1 ms.
- Search performed well for headphones and cameras but showed limitations for gaming mice, power banks, and charging cables.
- Cross-Encoder and metadata-aware reranking were tested but did not consistently improve relevance.

### Final Decision
Selected BM25 + E5 + RRF as the text search baseline and CLIP + FAISS for image search.

Future improvements include category filtering, better candidate retrieval, and relevance evaluation.

All search indexes are saved for integration into the ElectroLens application.

## Deployment

Save the search indexes and product metadata for the application. Load the encoders and indexes once at startup to enable faster searches.